# LSTM dự báo nhu cầu thuê xe theo thời gian

Notebook sử dụng dữ liệu đã tiền xử lý trong `data/processed`, giữ thứ tự thời gian và không shuffle khi huấn luyện.

- Target: `cnt`
- Cửa sổ thời gian: 24 quan sát trước đó
- Scaler chỉ fit trên tập train
- Đánh giá: RMSE, MAE, R²

In [1]:
# ========== BƯỚC 1: IMPORT THƯ VIỆN VÀ CẤU HÌNH ==========
from pathlib import Path  # Làm việc với đường dẫn file và thư mục
from time import perf_counter  # Đo thời gian huấn luyện
import os  # Cấu hình biến môi trường
import random  # Đặt seed cho Python

# Buộc TensorFlow chỉ sử dụng CPU để chạy ổn định trên các môi trường khác nhau.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np  # Xử lý mảng số học
import pandas as pd  # Đọc và xử lý dữ liệu dạng bảng
import tensorflow as tf  # Xây dựng và huấn luyện mô hình LSTM
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Tính metrics
from sklearn.preprocessing import StandardScaler  # Chuẩn hóa features và target

# Đặt seed để kết quả có thể tái lập giữa các lần chạy.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Tắt GPU nếu TensorFlow phát hiện GPU trước đó.
try:
    tf.config.set_visible_devices([], "GPU")
except RuntimeError:
    pass

# ========== TÌM THƯ MỤC GỐC VÀ CHUẨN BỊ ĐƯỜNG DẪN ==========
# Notebook có thể được chạy từ thư mục gốc hoặc từ thư mục notebooks.
project_root = Path.cwd()
if not (project_root / "data/processed/train.csv").exists():
    project_root = project_root.parent

processed_dir = project_root / "data/processed"  # Dữ liệu train/validation/test
metrics_dir = project_root / "results/metrics"  # Metrics của mô hình
models_dir = project_root / "models"  # File model đã huấn luyện
metrics_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root.resolve()}")
print(f"TensorFlow: {tf.__version__}")

Project root: E:\bike-demand-prediction-main
TensorFlow: 2.20.0


In [3]:
# ========== BƯỚC 2: LOAD VÀ CHUẨN BỊ CÁC TẬP DỮ LIỆU ==========
# Đọc một tập dữ liệu đã chia, parse ngày giờ và sắp xếp theo thời gian.
def load_split(name):
    return pd.read_csv(
        processed_dir / f"{name}.csv",
        parse_dates=["timestamp", "dteday"],
    ).sort_values("timestamp").reset_index(drop=True)

train_df = load_split("train")  # Dữ liệu dùng để huấn luyện
validation_df = load_split("validation")  # Dữ liệu dùng để chọn model tốt nhất
test_df = load_split("test")  # Dữ liệu đánh giá cuối cùng

# ========== BƯỚC 3: CHỌN TARGET, FEATURES VÀ CỬA SỔ THỜI GIAN ==========
target_column = "cnt"  # Số lượt thuê xe cần dự báo

# Loại bỏ target, định danh, ngày giờ và các cột được suy ra trực tiếp từ target.
excluded_columns = {
    target_column,
    "instant",
    "dteday",
    "timestamp",
    "casual",
    "registered",
}
feature_columns = [
    column for column in train_df.columns if column not in excluded_columns
]
sequence_length = 24  # LSTM sử dụng 24 quan sát trước để dự báo quan sát hiện tại

# Đảm bảo các tập dữ liệu tách theo thời gian, không bị đảo thứ tự hoặc leakage.
assert train_df["timestamp"].max() < validation_df["timestamp"].min()
assert validation_df["timestamp"].max() < test_df["timestamp"].min()
assert train_df[feature_columns].notna().all().all()
assert validation_df[feature_columns].notna().all().all()
assert test_df[feature_columns].notna().all().all()

# Fit scaler chỉ trên train; validation và test chỉ dùng transform.
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(train_df[feature_columns])
X_validation_scaled = feature_scaler.transform(validation_df[feature_columns])
X_test_scaled = feature_scaler.transform(test_df[feature_columns])

y_train_scaled = target_scaler.fit_transform(train_df[[target_column]]).ravel()
y_validation_scaled = target_scaler.transform(validation_df[[target_column]]).ravel()
y_test_scaled = target_scaler.transform(test_df[[target_column]]).ravel()

# Chuyển dữ liệu dạng bảng thành các chuỗi liên tiếp cho mô hình LSTM.
def make_sequences(features, targets, window):
    sequence_features = []
    sequence_targets = []
    for index in range(window, len(features)):
        sequence_features.append(features[index - window:index])
        sequence_targets.append(targets[index])
    return np.asarray(sequence_features), np.asarray(sequence_targets)

X_train_seq, y_train_seq = make_sequences(
    X_train_scaled, y_train_scaled, sequence_length
)

# Dùng phần cuối của tập trước làm context, nhưng không dùng target tương lai.
validation_context_X = np.vstack(
    [X_train_scaled[-sequence_length:], X_validation_scaled]
)
validation_context_y = np.concatenate(
    [y_train_scaled[-sequence_length:], y_validation_scaled]
)
X_validation_seq, y_validation_seq = make_sequences(
    validation_context_X, validation_context_y, sequence_length
)

test_context_X = np.vstack([X_validation_scaled[-sequence_length:], X_test_scaled])
test_context_y = np.concatenate([y_validation_scaled[-sequence_length:], y_test_scaled])
X_test_seq, y_test_seq = make_sequences(
    test_context_X, test_context_y, sequence_length
)

# Kiểm tra số chuỗi: train mất window đầu, validation/test giữ đủ số dòng.
assert len(X_train_seq) == len(train_df) - sequence_length
assert len(X_validation_seq) == len(validation_df)
assert len(X_test_seq) == len(test_df)

print(f"Features: {len(feature_columns)}")
print(f"Sequences: train={X_train_seq.shape}, validation={X_validation_seq.shape}, test={X_test_seq.shape}")

Features: 27
Sequences: train=(12023, 24, 27), validation=(2582, 24, 27), test=(2582, 24, 27)


In [ ]:
# ========== BƯỚC 4: XÂY DỰNG VÀ HUẤN LUYỆN MÔ HÌNH LSTM ==========
# LSTM đọc chuỗi 24 quan sát, sau đó dùng các lớp Dense để dự báo cnt.
lstm_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(sequence_length, len(feature_columns))
        ),
        tf.keras.layers.LSTM(32),  # Học các phụ thuộc theo thời gian trong chuỗi
        tf.keras.layers.Dense(16, activation="relu"),  # Lớp xử lý trung gian
        tf.keras.layers.Dense(1),  # Một giá trị đầu ra là nhu cầu dự báo
    ],
    name="bike_demand_lstm",
)

# Dùng MSE làm loss vì đây là bài toán hồi quy.
lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
)

# Dừng sớm nếu validation loss không còn cải thiện và khôi phục trọng số tốt nhất.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

start_time = perf_counter()
history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_validation_seq, y_validation_seq),
    epochs=30,
    batch_size=64,
    shuffle=False,  # Giữ thứ tự thời gian, tránh làm sai cấu trúc chuỗi
    callbacks=[early_stopping],
    verbose=1,
)
training_time = perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")
print(f"Epochs completed: {len(history.history['loss'])}")

Epoch 1/30


In [ ]:
# ========== BƯỚC 5: ĐÁNH GIÁ VÀ LƯU MÔ HÌNH ==========
# Dự báo trên một tập dữ liệu, đưa target và prediction về thang đo ban đầu.
def evaluate_split(split_name, y_true_scaled, X_sequence):
    predictions_scaled = lstm_model.predict(X_sequence, verbose=0).ravel()
    y_true = target_scaler.inverse_transform(
        y_true_scaled.reshape(-1, 1)
    ).ravel()
    predictions = target_scaler.inverse_transform(
        predictions_scaled.reshape(-1, 1)
    ).ravel()
    return {
        "model": "LSTM",
        "split": split_name,
        "RMSE": mean_squared_error(y_true, predictions) ** 0.5,  # Sai số bình phương trung bình căn bậc hai
        "MAE": mean_absolute_error(y_true, predictions),  # Sai số tuyệt đối trung bình
        "R2": r2_score(y_true, predictions),  # Mức độ giải thích biến thiên của target
        "Training Time": training_time,
    }

# Tính metrics riêng cho validation và test.
lstm_metrics = pd.DataFrame(
    [
        evaluate_split("validation", y_validation_seq, X_validation_seq),
        evaluate_split("test", y_test_seq, X_test_seq),
    ]
)

print("LSTM results:")
display(lstm_metrics.round(4))
lstm_metrics.to_csv(metrics_dir / "lstm_metrics.csv", index=False)

# Lưu model để có thể tải lại và dự báo mà không cần huấn luyện lại.
lstm_model.save(models_dir / "lstm.keras")
print(f"Saved model to: {models_dir / 'lstm.keras'}")

LSTM results:


,model,split,RMSE,MAE,R2,Training Time
0,LSTM,validation,69.9274,50.8008,0.9028,11.5304
1,LSTM,test,84.4831,59.5104,0.8441,11.5304


Saved model to: e:\bike-demand-prediction-main\models\lstm.keras


: 